## Comparing dataset projections (Part 2)

In [13]:
import ee
import geemap
ee.Authenticate()
ee.Initialize()

# Map Initialization
Map = geemap.Map()

# Panama boundary
countries = ee.FeatureCollection("FAO/GAUL/2015/level0")
panama_fc = countries.filter(ee.Filter.eq("ADM0_NAME", "Panama"))
panama_geom = panama_fc.geometry()

Map.centerObject(panama_geom, 7)

### Elevation (re-projection standard)

In [14]:
# DEM collection, ~30m resolution
dataset = (
    ee.ImageCollection('COPERNICUS/DEM/GLO30')
    .filterBounds(panama_geom)
)

# Create a single DEM image and clip it
elevation = (
    dataset
    .select('DEM')
    .first()
    .clip(panama_geom)
)

elevation_vis = {
    'min': 0,
    'max': 2500,
    'palette': ['0000ff', '00ffff', 'ffff00', 'ff0000', 'ffffff'],
}

Map.addLayer(elevation, elevation_vis, 'Panama DEM')

### Slope

In [15]:
# ~30 m resolution
slope = ee.Image("projects/deforestation-495419/assets/panama_slopee").clip(panama_geom)

# reprojecting to fit other data layers
sample_image = elevation
collection_projection = sample_image.projection()

reprojected_slope = (
    slope.resample("bilinear")
    .reproject(collection_projection)
    .rename("slope_reprojected")
    .clip(panama_geom)
)

# Map.addLayer(slope, {"min": 0, "max": 30}, "Slope")
Map.addLayer(reprojected_slope, {"min": 0, "max": 30}, "Reprojected Slope")

In [16]:
reprojected_slope.projection().getInfo()

{'type': 'Projection',
 'crs': 'EPSG:4326',
 'transform': [0.0002777777777777778,
  0,
  -78.00013888888888,
  0,
  -0.0002777777777777778,
  8.00013888888889]}

### Soils

In [17]:
# ~260m resolution
pa = ee.Image("projects/deforestation-495419/assets/Soil_Org_C").clip(panama_geom)
pn = ee.Image("projects/deforestation-495419/assets/Soil_N").clip(panama_geom)
ps = ee.Image("projects/deforestation-495419/assets/sand").clip(panama_geom)
ph = ee.Image("projects/deforestation-495419/assets/pH").clip(panama_geom)

pa_reprojected = pa.reproject(
    crs = elevation.projection(), scale = pa.projection().nominalScale()
)

pn_reprojected = pn.reproject(
    crs = elevation.projection(), scale = pn.projection().nominalScale()
)

ps_reprojected = ps.reproject(
    crs = elevation.projection(), scale = ps.projection().nominalScale()
)

ph_reprojected = ph.reproject(
    crs = elevation.projection(), scale = ph.projection().nominalScale()
)

Map.addLayer(pa_reprojected, {"min": 0, "max": 150}, "Soil Organic Carbon")
Map.addLayer(pn_reprojected, {"min": 0, "max": 80}, "Nitrogen")
Map.addLayer(ps_reprojected, {"min": 0, "max": 1000}, "Sand")
Map.addLayer(ph_reprojected, {"min": 40, "max": 80}, "pH")

In [18]:
pa_reprojected.projection().getInfo()
pn_reprojected.projection().getInfo()
ps_reprojected.projection().getInfo()
ph_reprojected.projection().getInfo()

{'type': 'Projection',
 'crs': 'EPSG:4326',
 'transform': [0.0023246840206374224,
  0,
  -78.00013888888888,
  0,
  0.0023246840206374224,
  8.00013888888889]}

In [19]:
pa_reprojected.projection().nominalScale().getInfo()

258.7826414326177

### Distance to deforested areas

In [20]:
# ~28m resolution
hansen = ee.Image("UMD/hansen/global_forest_change_2025_v1_13").clip(panama_geom)

lossyear = hansen.select("lossyear")

recent_loss = lossyear.gte(20).selfMask()

distance_loss = (
    recent_loss.fastDistanceTransform(256)
    .sqrt()
    .multiply(30)
    .rename("dist_loss")
    .clip(panama_geom)
)

# reprojecting to fit other data layers
sample_image = elevation
collection_projection = sample_image.projection()

reprojected_distance_loss = (
    distance_loss.resample("bilinear")
    .reproject(collection_projection)
    .rename("distance_loss_reprojected")
    .clip(panama_geom)
)

# Map.addLayer(recent_loss, {"palette": ["red"]}, "Recent Loss")
Map.addLayer(reprojected_distance_loss, {"min": 0, "max": 5000}, "Distance to Loss")

In [21]:
reprojected_distance_loss.projection().getInfo()

KeyboardInterrupt: 

### Precipitation (monthly)

In [36]:
# ~5566m resolution
chirps = (
    ee.ImageCollection("UCSB-CHC/CHIRPS/V3/DAILY_SAT")
    .filterDate("2018-05-01", "2018-05-31")
    .first()
    .select("precipitation")
)

# precip = chirps.mean().clip(panama_geom) --> lowers resolution BIG TIME (~111,319m)

chirps_reprojected = chirps.reproject(
    crs = elevation.projection(), scale = chirps.projection().nominalScale()
)

Map.addLayer(
    chirps_reprojected,
    {
        "min": 1,
        "max": 17,
        "palette": ["#001137", "#0aab1e", "#e7eb05", "#2c7fb8", "#253494"]
    },
    "Precipitation"
)

In [46]:
chirps.projection().nominalScale().getInfo()

5565.974622603162

In [38]:
chirps_reprojected.projection().nominalScale().getInfo()

5565.974622603162

### Temperature (monthly)

In [34]:
# ~11,131m resolution
temp = (
    ee.ImageCollection("ECMWF/ERA5_LAND/MONTHLY_AGGR")
    .filterDate("2023-01-01", "2023-02-01")
    .first()
    .clip(panama_geom)
)

temp_reprojected = temp.reproject(
    crs = elevation.projection(), scale = temp.projection().nominalScale()
)

Map.addLayer(
    temp_reprojected,
    {
        "bands": ["temperature_2m"],
        "min": 250,
        "max": 320
    },
    "Temperature"
)

In [35]:
temp_reprojected.projection().getInfo()

{'type': 'Projection',
 'crs': 'EPSG:4326',
 'transform': [0.10000000000000002,
  0,
  -78.00013888888888,
  0,
  0.10000000000000002,
  8.00013888888889]}

### Distance to hydrographic basins

In [39]:
# 100m resolution
basins = ee.Image("projects/deforestation-495419/assets/hydrographic_basins").clip(panama_geom)

basins_int = basins.toInt()
neighbours = basins_int.focal_mode(1)

edges = basins_int.neq(neighbours).selfMask()

dist_basin = (
    edges.fastDistanceTransform(512).sqrt()
    .multiply(basins.projection().nominalScale())
    .clip(panama_geom)
)

# reprojecting to fit other data layers
sample_image = elevation
collection_projection = sample_image.projection()

reprojected_distance_basin = (
    dist_basin.resample("bilinear")
    .reproject(collection_projection)
    .rename("distance_loss_reprojected")
    .clip(panama_geom)
)

# Map.addLayer(basins, {}, "Basins")
# Map.addLayer(edges, {"palette": ["red"]}, "Basin Edges")
# Map.addLayer(dist_basin, {"min": 0, "max": 50000}, "Distance Basin")
Map.addLayer(reprojected_distance_basin, {"min": 0, "max": 50000}, "Distance to HydrographicBasin")

In [41]:
reprojected_distance_basin.projection().getInfo()

{'type': 'Projection',
 'crs': 'EPSG:4326',
 'transform': [0.0002777777777777778,
  0,
  -78.00013888888888,
  0,
  -0.0002777777777777778,
  8.00013888888889]}

### Generate map

In [ ]:
Map

Map(bottom=15906.0, center=[8.5158389458998, -80.10966640141521], controls=(WidgetControl(options=['position',…